# BookNLP on Colab — batch over a Gutenberg ID list

Adapted from the single-book *Mrs. Dalloway* notebook. Same environment, same
BookNLP settings; everything downstream of the install now loops.

**Before you start:** `Runtime` -> `Change runtime type` -> Hardware accelerator = **T4 GPU**.

### Output layout

```
Drive/booknlp_batch/output/<gutenberg_id>/<gutenberg_id>.tokens.gz
                                          <gutenberg_id>.entities.gz
                                          <gutenberg_id>.quotes.gz
                                          <gutenberg_id>.supersense.gz
                                          <gutenberg_id>.book
                                          _COMPLETE       <- written only on success
```

### How to split the run across two machines

Set `N_SHARDS = 2` in Cell 3. You run with `SHARD = 1`, your friend runs the
**same notebook and same CSV** with `SHARD = 2`. Books are assigned round-robin,
so the shards get a similar mix of long and short books. Nothing else changes,
and the two of you can merge results at the end with Cell 9.

### Cleaning is deliberately minimal

Strip the Project Gutenberg header/footer, normalise line endings and Unicode,
drop `_italic_` markers, collapse runs of blank lines. That is all. Front matter,
chapter headings, and section dividers are left in place for your downstream
pipeline to handle. Nothing here removes narrative text.

## Cell 1 — Check the runtime

In [1]:
!python --version
!nvidia-smi -L || echo "No GPU attached - it will still run, just slower"

Python 3.12.13
GPU 0: Tesla T4 (UUID: GPU-209564ea-2f30-1b70-e156-bd5421032d98)


## Cell 2 — Build a Python 3.10 environment and install BookNLP

Unchanged from your single-book notebook.

Why not just `pip install booknlp`: BookNLP's `setup.py` pins `transformers<=4.30.0`.
That version needs `tokenizers 0.13.3`, which only ships wheels up to **cp311** — on Colab's
Python 3.12 pip falls back to compiling Rust from source and fails. So we build an isolated
Python 3.10 virtualenv with `uv` and install BookNLP there.

`--seed` matters: it puts `setuptools` inside the venv, otherwise BookNLP's use of
`pkg_resources` raises `ModuleNotFoundError`.

This cell takes 3-5 minutes.

In [2]:
!pip install -q uv

# Create a Python 3.10 venv (uv downloads a standalone CPython 3.10 automatically)
!uv venv /content/bnlp --python 3.10 --seed

PY = "/content/bnlp/bin/python"

# numpy<2   : transformers 4.30 is not compatible with numpy 2.x
# torch 2.4.1 : the linux wheel on PyPI bundles CUDA, so Colab's GPU works out of the box
!uv pip install --python {PY} "numpy<2" "torch==2.4.1" "spacy==3.7.5" \
    "transformers==4.30.0" "tokenizers<0.14" "huggingface_hub<0.20" click booknlp
# spaCy English model (BookNLP uses it for POS tagging, dependency parsing, sentence splitting)
!{PY} -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.0/22.0 MB 64.6 MB/s eta 0:00:00
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment with seed packages at: bnlp
 + packaging==26.2
 + pip==26.2
 + setuptools==83.0.0
 + wheel==0.47.0
Activate with: source bnlp/bin/activate
Using Python 3.10.12 environment at: bnlp
Resolved 86 packages in 1.39s
Prepared 83 packages in 1m 27s
Installed 83 packages in 1.12s
 + absl-py==2.5.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + astunparse==1.6.3
 + blis==0.7.11
 + booknlp==1.0.8
 + catalogue==2.0.10
 + certifi==2026.7.22
 + charset-normalizer==3.4.9
 + click==8.4.2
 + cloudpathlib==0.24.0
 + confection==0.1.5
 + cymem==2.0.13
 + filelock==3.32.2
 + flatbuffers==25.12.19
 + fsspec==2026.7.0
 + gast==0.7.0
 + google-pasta==0.2.0
 + grpcio==1.83.0
 + h5py==3.14.0
 + huggingface-hub==0.19.4
 + idna==3.18
 + jinja2==3.1.6
 + keras==3.12.4
 + langcodes==3.5.1
 + libclang==18.1.1
 + markdown-it-py==4.2.0
 + markupsa

In [3]:
# Re-pin: installing the spaCy model can drag transformers/tokenizers back up.
# The setuptools pin your run cell needed is folded in here so it happens once.
PY = "/content/bnlp/bin/python"
!uv pip install --python {PY} "transformers==4.30.0" "tokenizers<0.14" "huggingface_hub<0.20" "setuptools<82"
!uv pip uninstall --python {PY} tensorflow

Using Python 3.10.12 environment at: bnlp
Resolved 18 packages in 401ms
Prepared 1 package in 193ms
Uninstalled 1 package in 8ms
Installed 1 package in 10ms
 - setuptools==83.0.0
 + setuptools==81.0.0
Using Python 3.10.12 environment at: bnlp
Uninstalled 1 package in 681ms
 - tensorflow==2.21.0


In [4]:
PY = "/content/bnlp/bin/python"
!uv pip install --python {PY} click
!{PY} -m spacy download en_core_web_sm

Using Python 3.10.12 environment at: bnlp
Checked 1 package in 7ms
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 23.9 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [5]:
%%writefile /content/verify.py
import torch, transformers, spacy, booknlp, pkg_resources
print("torch       ", torch.__version__, "| cuda:", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("spacy       ", spacy.__version__)
nlp = spacy.load("en_core_web_sm")
print("en_core_web_sm OK |", [(t.text, t.pos_) for t in nlp("She wondered if he knew.")][:3])
print("pkg_resources OK")

Writing /content/verify.py


### Verify the install

In [6]:
%%writefile /content/verify.py
import torch, transformers, spacy, booknlp, pkg_resources
print("torch       ", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("spacy       ", spacy.__version__)
print("pkg_resources OK")

Overwriting /content/verify.py


In [7]:
PY = "/content/bnlp/bin/python"
!{PY} /content/verify.py

/content/verify.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import torch, transformers, spacy, booknlp, pkg_resources
torch        2.4.1+cu121 | cuda available: True
transformers 4.30.0
spacy        3.7.5
pkg_resources OK


## Cell 3 — Configuration

### Drive

`ROOT` lives on Drive, and this is what stops a disconnect from costing you the
run. But note **how** the writing works, because it matters:

BookNLP writes each book to a *local* scratch directory (`/content/work/`), and
only once `process()` returns cleanly does the finished folder get copied to
Drive and stamped `_COMPLETE`. Writing BookNLP's output straight to Drive would
mean hundreds of small writes through a FUSE mount — slow, and if the session
dies mid-write you get a half-written folder that looks finished. This way a
book is either complete on Drive or absent, never in between.

Cleaned source texts also live on Drive, so a reconnect does not re-download.

### Sharding

`SHARD` / `N_SHARDS` split the book list round-robin. Same notebook, same CSV,
one number different per person. `SHARD = 1, N_SHARDS = 1` runs everything.

### Size

The `.tokens` file for a novel runs 5-25 MB, and there are four TSVs per book.
Uncompressed, 353 books can reach 10+ GB — more than a free Drive account holds.
So the TSVs are gzipped on the way to Drive (pandas reads `.gz` transparently,
no change to your downstream code) and the `.book.html` browser render is
dropped. Set either flag to `False` if you want the raw files.

In [8]:
# --- who is running which part ------------------------------------------------
SHARD    = 1        # you: 1     your friend: 2
N_SHARDS = 1        # set to 1 to run the whole list on one machine

RUN_TAG = 'default_run' # Add this line to define RUN_TAG

# --- storage ------------------------------------------------------------------
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/Interiority_26Spring/Data/booknlp_batch"
else:
    ROOT = "/content/booknlp_batch"

import os
CLEAN_DIR  = os.path.join(ROOT, "clean")          # cleaned .txt, one per book
OUTPUT_DIR = os.path.join(ROOT, "output")         # final output, one folder per book
WORK_DIR   = "/content/work"                      # local scratch, never on Drive
MANIFEST  = os.path.join(ROOT, f"manifest_{RUN_TAG}.csv")
BATCH_LOG = os.path.join(ROOT, f"batch_log_{RUN_TAG}.csv")
for d in (CLEAN_DIR, OUTPUT_DIR, WORK_DIR):
    os.makedirs(d, exist_ok=True)

# --- BookNLP settings (same as the single-book notebook) ----------------------
# supersense gives you verb.cognition / verb.emotion / noun.cognition, so keep it
# even though it is the slow part.
PIPELINE = "entity,quote,supersense,event,coref"
MODEL    = "big"          # "small" is ~2x faster and noticeably worse at coref

# --- output size --------------------------------------------------------------
GZIP_TSV    = True        # .tokens -> .tokens.gz ; pandas reads it transparently
DROP_HTML   = True        # discard the .book.html render

# --- download politeness ------------------------------------------------------
DOWNLOAD_DELAY = 1.5
MAX_RETRIES    = 3

print(f"shard {SHARD} of {N_SHARDS}")
print("output ->", OUTPUT_DIR)
print("scratch->", WORK_DIR)

Mounted at /content/drive
shard 1 of 1
output -> /content/drive/MyDrive/Interiority_26Spring/Data/booknlp_batch/output
scratch-> /content/work


## Cell 4 — Load your CSV and take this shard

Run this and pick your CSV from the file dialog, or set `CSV_PATH` to a path on
Drive and skip the upload. **Both of you use the same unsplit CSV** — the shard
filter below is what divides the work.

The ID column is auto-detected (`gutenberg_id`, `gid`, `pg_id`, `id`, ...).
Optional columns used if present: `title`, `author`.

In [9]:
BOOK_IDS = [
    21571, 72680, 71156, 67669, 69476, 78686, 74233, 71357, 77747, 75875,
    77588, 73376, 67679, 74687, 76222, 76352, 79003, 11635, 68790, 68569,
    23419, 77782, 76205, 66837, 73673, 71761, 75340, 70582, 73734, 66528,
    68858, 68789, 73672, 75410, 72678, 79040, 67280, 68033, 68936, 71360,
    73473, 79006,
]

books, seen = [], set()
for g in BOOK_IDS:
    gid = str(g).strip()
    if gid in seen:
        continue
    seen.add(gid)
    books.append({"gutenberg_id": gid, "title": "", "author": ""})

print(f"{len(books)} 本待跑")
print(books[0]["gutenberg_id"], "...", books[-1]["gutenberg_id"])

42 本待跑
21571 ... 79006


Optional — if you would rather hand your friend a CSV than ask them to change
`SHARD`, this writes one file per shard to Drive. They then upload their file in
Cell 4 with `SHARD = 1, N_SHARDS = 1`.

In [10]:
# for s in range(1, N_SHARDS + 1):
#     sub = pd.DataFrame([b for i, b in enumerate(all_books) if i % N_SHARDS == s - 1])
#     path = os.path.join(ROOT, f"books_shard{s}of{N_SHARDS}.csv")
#     sub.to_csv(path, index=False)
#     print(len(sub), "books ->", path)

## Cell 5 — Download and clean

Deliberately conservative. Only two things get removed, and neither is narrative
text:

1. **The Project Gutenberg header and footer**, matched on the standard
   `*** START OF THE PROJECT GUTENBERG EBOOK ... ***` markers. If a marker is
   missing the text is kept whole and flagged in the manifest rather than cut on
   a guess.
2. **`_italic_` underscore markers**, because spaCy tokenizes `_that_` oddly.
   The word survives, only the underscores go.

Then line endings are normalised, runs of 3+ blank lines collapse to one blank
line, and Unicode is NFC-normalised.

Front matter, title pages, tables of contents, chapter headings, and `* * *`
dividers all stay — your pipeline handles those later. The one consequence worth
knowing: they consume `paragraph_ID`s, so paragraph 0 is often not the first
paragraph of the novel.

Downloads try `aleph.gutenberg.org` and `gutenberg.pglaf.org` before
`www.gutenberg.org` — PG asks bulk downloaders to use mirrors, and Colab's
shared IP is exactly the kind that gets blocked. Already-cleaned books are
skipped, so re-running costs nothing.

In [11]:
import re, time, unicodedata, urllib.request, urllib.error, csv, os, random

UA = "Mozilla/5.0 (research; digital humanities corpus build)"

import ssl
DEAD_HOSTS = set()

def mirror_path(gid):
    """Gutenberg mirror layout: 1342 -> 1/3/4/1342/"""
    return ("0/" if len(gid) == 1 else "/".join(gid[:-1]) + "/") + gid + "/"

def candidate_urls(gid):
    mp = mirror_path(gid)
    return [
        # pglaf 镜像优先（PG 要求批量下载走镜像），aleph 已移除：证书主机名不匹配
        f"https://gutenberg.pglaf.org/{mp}{gid}-0.txt",
        f"https://gutenberg.pglaf.org/{mp}{gid}.txt",
        # 主站兜底
        f"https://www.gutenberg.org/cache/epub/{gid}/pg{gid}.txt",
        f"https://www.gutenberg.org/files/{gid}/{gid}-0.txt",
    ]

def fetch(gid):
    from urllib.parse import urlparse
    last = None
    for url in candidate_urls(gid):
        host = urlparse(url).netloc
        if host in DEAD_HOSTS:
            continue
        for attempt in range(MAX_RETRIES):
            try:
                req = urllib.request.Request(url, headers={"User-Agent": UA})
                with urllib.request.urlopen(req, timeout=20) as r:
                    data = r.read()
                if len(data) < 1000:
                    last = f"too short ({len(data)}b)"; break
                for enc in ("utf-8-sig", "utf-8", "latin-1"):
                    try:
                        return data.decode(enc), url
                    except UnicodeDecodeError:
                        continue
                last = "undecodable"; break

            except urllib.error.HTTPError as e:
                last = f"HTTP {e.code}"
                if e.code in (404, 403):        # 路径不对/被拒，换下一个 URL
                    break
                if attempt < MAX_RETRIES - 1:   # 最后一次不再 sleep
                    time.sleep(2 ** attempt)

            except (ssl.SSLError, urllib.error.URLError) as e:
                # 证书错误 / DNS / 连接被拒：主机级别的永久故障，标记后整批跳过
                reason = getattr(e, "reason", e)
                if isinstance(e, ssl.SSLError) or isinstance(reason, (ssl.SSLError, OSError)):
                    DEAD_HOSTS.add(host)
                    print(f"    [{host} unreachable: {reason} - skipping for the rest of the run]",
                          flush=True)
                    last = str(reason)
                    break
                last = str(reason)
                if attempt < MAX_RETRIES - 1:
                    time.sleep(2 ** attempt)

            except Exception as e:
                last = f"{type(e).__name__}: {e}"
                if attempt < MAX_RETRIES - 1:
                    time.sleep(2 ** attempt)

    raise RuntimeError(f"could not download {gid} ({last})")

START_RE = re.compile(r"\*\*\*\s*START OF TH(E|IS) PROJECT GUTENBERG.*?\*\*\*", re.S)
END_RE   = re.compile(r"\*\*\*\s*END OF TH(E|IS) PROJECT GUTENBERG.*?\*\*\*", re.S)

def clean(raw):
    """Minimal, non-destructive. Returns (text, notes)."""
    notes = []
    raw = raw.replace("\r\n", "\n").replace("\r", "\n")

    m = START_RE.search(raw)
    if m:
        raw = raw[m.end():]
    else:
        notes.append("no_start_marker")

    m = END_RE.search(raw)
    if m:
        raw = raw[:m.start()]
    else:
        notes.append("no_end_marker")

    raw = re.sub(r"_([^_\n]+)_", r"\1", raw)      # italic markers, keeps the word
    raw = re.sub(r"[ \t]+\n", "\n", raw)          # trailing whitespace
    raw = re.sub(r"\n{3,}", "\n\n", raw)          # 3+ blank lines -> one
    raw = unicodedata.normalize("NFC", raw).strip() + "\n"
    return raw, ",".join(notes)

rows, failures = [], []
for n, b in enumerate(books, 1):
    gid  = b["gutenberg_id"]
    path = os.path.join(CLEAN_DIR, f"{gid}.txt")

    if os.path.exists(path) and os.path.getsize(path) > 1000:
        txt = open(path, encoding="utf-8").read()
        rows.append({**b, "clean_path": path, "chars": len(txt), "words": len(txt.split()),
                     "paragraphs": txt.count("\n\n") + 1, "notes": "cached", "source_url": ""})
        print(f"[{n}/{len(books)}] {gid} cached", flush=True)
        continue

    try:
        raw, url = fetch(gid)
        txt, notes = clean(raw)
        if len(txt) < 5000:
            raise RuntimeError(f"suspiciously short after cleaning: {len(txt)} chars")
        with open(path, "w", encoding="utf-8") as f:
            f.write(txt)
        rows.append({**b, "clean_path": path, "chars": len(txt), "words": len(txt.split()),
                     "paragraphs": txt.count("\n\n") + 1, "notes": notes, "source_url": url})
        print(f"[{n}/{len(books)}] {gid}  {len(txt.split()):>7,} words"
              f"{'  [' + notes + ']' if notes else ''}", flush=True)
    except Exception as e:
        failures.append((gid, str(e)))
        print(f"[{n}/{len(books)}] {gid}  FAILED: {e}", flush=True)

    time.sleep(DOWNLOAD_DELAY + random.random())

with open(MANIFEST, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)

print(f"\n{len(rows)} ready, {len(failures)} failed -> {MANIFEST}")
for gid, err in failures:
    print("  ", gid, err)

[1/42] 21571 cached
[2/42] 72680 cached
[3/42] 71156 cached
[4/42] 67669 cached
[5/42] 69476 cached
[6/42] 78686 cached
[7/42] 74233 cached
[8/42] 71357 cached
[9/42] 77747 cached
[10/42] 75875 cached
[11/42] 77588 cached
[12/42] 73376 cached
[13/42] 67679 cached
[14/42] 74687 cached
[15/42] 76222 cached
[16/42] 76352 cached
[17/42] 79003 cached
[18/42] 11635 cached
[19/42] 68790 cached
[20/42] 68569 cached
[21/42] 23419 cached
[22/42] 77782 cached
[23/42] 76205 cached
[24/42] 66837 cached
[25/42] 73673 cached
[26/42] 71761 cached
[27/42] 75340 cached
[28/42] 70582 cached
[29/42] 73734 cached
[30/42] 66528 cached
[31/42] 68858 cached
[32/42] 68789 cached
[33/42] 73672 cached
[34/42] 75410 cached
[35/42] 72678 cached
[36/42] 79040 cached
[37/42] 67280 cached
[38/42] 68033 cached
[39/42] 68936 cached
[40/42] 71360 cached
[41/42] 73473 cached
[42/42] 79006 cached

42 ready, 0 failed -> /content/drive/MyDrive/Interiority_26Spring/Data/booknlp_batch/manifest_default_run.csv


## Cell 6 — Quick sanity check

Not looking for clean openings — front matter is expected. Looking for the two
things that would waste GPU hours: a book whose length is wildly off (wrong
edition, a stub, an abridgement) and a book where the PG markers were missing,
which usually means the license text is still attached.

In [13]:
import pandas as pd

In [14]:
man = pd.read_csv(MANIFEST, dtype={"gutenberg_id": str}).fillna("")

print(f"{len(man)} books, {man.words.sum():,} words total")
print("\nword counts:")
print(man.words.describe().to_string())

flag = man[man.notes.str.contains("marker", na=False)]
print(f"\n{len(flag)} books missing a PG marker (check these):")
if len(flag):
    print(flag[["gutenberg_id", "title", "notes"]].to_string(index=False))

print("\nshortest / longest:")
print(man.nsmallest(3, "words")[["gutenberg_id", "title", "words"]].to_string(index=False))
print(man.nlargest(3, "words")[["gutenberg_id", "title", "words"]].to_string(index=False))

print("\nsample openings (front matter here is fine):")
for _, r in man.sample(min(4, len(man)), random_state=0).iterrows():
    print("-" * 70)
    print(f"{r.gutenberg_id} {r.title}  {r.words:,} words")
    print(open(r.clean_path, encoding="utf-8").read()[:200].replace("\n", " "))

42 books, 2,874,259 words total

word counts:
count        42.000000
mean      68434.738095
std       32742.384893
min       13089.000000
25%       48257.750000
50%       62616.000000
75%       85668.750000
max      158164.000000

0 books missing a PG marker (check these):

shortest / longest:
gutenberg_id title  words
       75340        13089
       23419        18135
       67280        22575
gutenberg_id title  words
       66528       158164
       21571       139416
       77747       127860

sample openings (front matter here is fine):
----------------------------------------------------------------------
68858   83,055 words
LADY BARBARITY    A ROMANCE    BY   J. C. SNAITH    AUTHOR OF MISTRESS DOROTHY MARVIN   AND FIERCEHEART, THE SOLDIER    NEW YORK   D. APPLETON AND COMPANY   1899    COPYRIGHT, 1898, 1899,   BY D. APPL
----------------------------------------------------------------------
67280   22,575 words
THE HISTORY OF THE   LADY BETTY STAIR  [Illustration: “‘I present 

## Cell 7 — The batch runner

The one structural change from your notebook: **`BookNLP()` is constructed once,
outside the loop.** Instantiating it per book would reload ~1.5 GB of weights on
every iteration.

Per book: process into local scratch -> gzip the TSVs -> copy to Drive -> stamp
`_COMPLETE` -> delete scratch. A book interrupted at any point leaves no
`_COMPLETE`, so the next run redoes it rather than trusting a partial folder.
`BNLP_HOURS` stops the loop cleanly before a session limit instead of being
killed mid-book.

**First run downloads ~1.5 GB of BookNLP weights** into `/root/booknlp_models/`,
over plain HTTP from `ischool.berkeley.edu`; it occasionally stalls. If it hangs
at the start, interrupt, `!rm -rf /root/booknlp_models`, re-run.

In [15]:
%%writefile /content/run_booknlp_batch.py
import csv, gc, gzip, os, shutil, sys, time, traceback

MANIFEST, OUTPUT_DIR, WORK_DIR, LOG_PATH = sys.argv[1:5]

PIPELINE  = os.environ.get("BNLP_PIPELINE", "entity,quote,supersense,event,coref")
MODEL     = os.environ.get("BNLP_MODEL", "big")
BUDGET    = float(os.environ.get("BNLP_HOURS", "0")) * 3600     # 0 = no limit
GZIP_TSV  = os.environ.get("BNLP_GZIP", "1") == "1"
DROP_HTML = os.environ.get("BNLP_DROP_HTML", "1") == "1"

import torch
from booknlp.booknlp import BookNLP

TSV = (".tokens", ".entities", ".quotes", ".supersense")

def publish(workdir, destdir):
    """Copy a finished book to Drive, compressing on the way. Marker written last."""
    tmp = destdir + ".partial"
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp, exist_ok=True)
    for fn in sorted(os.listdir(workdir)):
        src = os.path.join(workdir, fn)
        if DROP_HTML and fn.endswith(".html"):
            continue
        if GZIP_TSV and fn.endswith(TSV):
            with open(src, "rb") as fi, gzip.open(os.path.join(tmp, fn + ".gz"), "wb", 6) as fo:
                shutil.copyfileobj(fi, fo, 1024 * 1024)
        else:
            shutil.copy2(src, os.path.join(tmp, fn))
    shutil.rmtree(destdir, ignore_errors=True)
    os.rename(tmp, destdir)
    open(os.path.join(destdir, "_COMPLETE"), "w").close()

with open(MANIFEST, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

todo = [r for r in rows
        if not os.path.exists(os.path.join(OUTPUT_DIR, r["gutenberg_id"], "_COMPLETE"))]

print(f"{len(rows)} in manifest | {len(rows) - len(todo)} already complete | "
      f"{len(todo)} to run", flush=True)
print(f"model={MODEL}  pipeline={PIPELINE}  cuda={torch.cuda.is_available()}  "
      f"gzip={GZIP_TSV}", flush=True)
if not todo:
    print("nothing to do"); sys.exit(0)

t0 = time.time()
booknlp = BookNLP("en", {"pipeline": PIPELINE, "model": MODEL})
print(f"models loaded in {time.time() - t0:.0f}s", flush=True)

new_log = not os.path.exists(LOG_PATH)
log  = open(LOG_PATH, "a", newline="", encoding="utf-8")
logw = csv.writer(log)
if new_log:
    logw.writerow(["gutenberg_id", "status", "seconds", "words", "error", "finished_at"])
    log.flush()

start, ok, err = time.time(), 0, 0

for i, r in enumerate(todo, 1):
    gid   = r["gutenberg_id"]
    words = r.get("words", "")
    work  = os.path.join(WORK_DIR, gid)
    dest  = os.path.join(OUTPUT_DIR, gid)

    if BUDGET and (time.time() - start) > BUDGET:
        print(f"\ntime budget reached - stopping cleanly, {len(todo) - i + 1} books "
              f"left. Re-run this cell to continue.", flush=True)
        break

    print(f"[{i}/{len(todo)}] {gid} ({words} words) ...", end=" ", flush=True)
    t = time.time()
    try:
        shutil.rmtree(work, ignore_errors=True)
        os.makedirs(work, exist_ok=True)
        booknlp.process(r["clean_path"], work, gid)
        publish(work, dest)
        dt = time.time() - t; ok += 1
        print(f"done in {dt/60:.1f} min", flush=True)
        logw.writerow([gid, "ok", round(dt, 1), words, "", time.strftime("%F %T")])
    except Exception as e:
        dt = time.time() - t; err += 1
        print(f"FAILED after {dt/60:.1f} min: {type(e).__name__}: {e}", flush=True)
        traceback.print_exc()
        logw.writerow([gid, "failed", round(dt, 1), words,
                       f"{type(e).__name__}: {e}", time.strftime("%F %T")])
    finally:
        shutil.rmtree(work, ignore_errors=True)
        log.flush()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    el = time.time() - start
    print(f"    elapsed {el/3600:.2f}h | avg {el/i/60:.1f} min/book | "
          f"eta {(len(todo)-i)*el/i/3600:.1f}h", flush=True)

log.close()
print(f"\nBATCH DONE  ok={ok}  failed={err}", flush=True)

Writing /content/run_booknlp_batch.py


Run it. This cell sits there for hours by design — it prints a line per book
with a running ETA. On disconnect: reconnect, re-run Cells 1-3, then this cell.
It resumes from Drive.

In [16]:
PY = "/content/bnlp/bin/python"
import os
os.environ["BNLP_PIPELINE"]  = PIPELINE
os.environ["BNLP_MODEL"]     = MODEL
os.environ["BNLP_HOURS"]     = "11"                     # "0" for no limit
os.environ["BNLP_GZIP"]      = "1" if GZIP_TSV  else "0"
os.environ["BNLP_DROP_HTML"] = "1" if DROP_HTML else "0"

!{PY} /content/run_booknlp_batch.py "{MANIFEST}" "{OUTPUT_DIR}" "{WORK_DIR}" "{BATCH_LOG}"

/content/bnlp/lib/python3.10/site-packages/booknlp/booknlp.py:50: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if __name__ is "__main__":
/content/bnlp/lib/python3.10/site-packages/booknlp/english/entity_tagger.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
using device cuda
42 in manifest | 0 already complete | 42 to run
model=big  pipeline=entity,quote,supersense,event,coref  cuda=True  gzip=True
{'pipeline': 'entity,quote,supersense,event,coref', 'model': 'big'}
downloading entities_google_bert_uncased_L-6_H-768_A-12-v1.0.model
downloading coref_google_bert_uncased_L-12_H-768_A-12-v1.0.model
downloading speaker_google_bert_uncased_L-12_H-768_A-12-v1.0.1.model



/content/bnlp/lib/python3.10/site-packages/transformers/modeling_utils.py:463: FutureWarni

## Cell 8 — Progress and failures

In [17]:
import pandas as pd, os, glob

done = {os.path.basename(os.path.dirname(p))
        for p in glob.glob(os.path.join(OUTPUT_DIR, "*", "_COMPLETE"))}
man  = pd.read_csv(MANIFEST, dtype={"gutenberg_id": str})
mine = [g for g in man.gutenberg_id if g in done]
print(f"this shard: {len(mine)} / {len(man)} complete")
print(f"in output folder overall (incl. merged): {len(done)}")

missing = [g for g in man.gutenberg_id if g not in done]
print(f"\n{len(missing)} remaining:", missing[:20], "..." if len(missing) > 20 else "")

if os.path.exists(BATCH_LOG):
    log = pd.read_csv(BATCH_LOG, dtype={"gutenberg_id": str})
    print("\nstatus:"); print(log.status.value_counts().to_string())
    okr = log[log.status == "ok"]
    if len(okr):
        print(f"median {okr.seconds.median()/60:.1f} min/book | "
              f"projected for {len(missing)} left: {okr.seconds.median()*len(missing)/3600:.1f}h")
    bad = log[log.status == "failed"]
    if len(bad):
        print("\nfailures:")
        print(bad[["gutenberg_id", "error"]].to_string(index=False))

# retry a failure by deleting its folder, then re-run Cell 7:
# import shutil; shutil.rmtree(os.path.join(OUTPUT_DIR, "12345"), ignore_errors=True)

this shard: 42 / 42 complete
in output folder overall (incl. merged): 168

0 remaining: [] 

status:
status
ok    42
median 2.1 min/book | projected for 0 left: 0.0h


## Cell 10 — Load the output

Runs in Colab's default Python 3.12 with pandas; the venv is not needed since
BookNLP writes plain TSV and JSON. `read_csv` decompresses `.gz` on its own, so
this is the same code you would write against uncompressed output.

Two things about `.tokens` that the README gets wrong:

- The README lists 12 columns; the code actually writes **13** (there is an extra `fine_POS_tag`).
- `byte_onset` / `byte_offset` are misnamed. The source stores spaCy's `tok.idx`, which is a
  **character** offset. So slice the original text directly with `text[onset:offset]` —
  do not `.encode()` first.

`quoting=3` (QUOTE_NONE) is essential — otherwise the novel's quotation marks swallow whole rows.

In [18]:
import pandas as pd, os, glob

def load_book(gid, exts=(".tokens", ".entities", ".quotes", ".supersense")):
    """Load one book's BookNLP output. Handles gzipped and plain files."""
    gid = str(gid)
    out = {}
    for e in exts:
        p = os.path.join(OUTPUT_DIR, gid, gid + e)
        p = p if os.path.exists(p) else p + ".gz"
        out[e.lstrip(".")] = pd.read_csv(p, sep="\t", quoting=3, keep_default_na=False)
    return out

gid = sorted(done)[0]
d = load_book(gid)
for k, v in d.items():
    print(f"{k:<11}", v.shape)
print("columns:", list(d["tokens"].columns))
print("paragraphs:", d["tokens"].paragraph_ID.nunique())
d["tokens"].head(10)

tokens      (21977, 13)
entities    (2273, 6)
quotes      (212, 7)
supersense  (6208, 4)
columns: ['paragraph_ID', 'sentence_ID', 'token_ID_within_sentence', 'token_ID_within_document', 'word', 'lemma', 'byte_onset', 'byte_offset', 'POS_tag', 'fine_POS_tag', 'dependency_relation', 'syntactic_head_ID', 'event']
paragraphs: 263


,paragraph_ID,sentence_ID,token_ID_within_sentence,token_ID_within_document,word,lemma,byte_onset,byte_offset,POS_tag,fine_POS_tag,dependency_relation,syntactic_head_ID,event
0,0,0,0,0,THE,the,0,3,DET,DT,det,3,O
1,0,0,1,1,WENDIGO,WENDIGO,4,11,PROPN,NNP,compound,3,O
2,1,0,2,2,Algernon,Algernon,13,21,PROPN,NNP,compound,3,O
3,1,0,3,3,Blackwood,Blackwood,22,31,PROPN,NNP,compound,5,O
4,2,0,4,4,1910,1910,33,37,NUM,CD,appos,3,O
5,3,0,5,5,I,I,39,40,PRON,PRP,nsubj,12,O
6,4,0,6,6,A,a,42,43,DET,DT,det,8,O
7,4,0,7,7,considerable,considerable,44,56,ADJ,JJ,amod,8,O
8,4,0,8,8,number,number,57,63,NOUN,NN,nsubj,12,O
9,4,0,9,9,of,of,64,66,ADP,IN,prep,8,O
